In [7]:
import os
import yaml
import pandas as pd

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Mobile App Data\Config Files"
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "7.1-YML_List.csv")

# === TEST CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    'Firebase_Full': ['gcloud firebase test android run'],
    'Firebase_Compact': ['Firebase-Test-Lab-Action'],
    'Appcenter': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'Browserstack': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact': ['malinskiy/action-android/emulator-run-cmd'],
    'emulator_manual': ['create avd'],
    'GitHub_GMD': ['cleanManagedDevices', 'ManagedVirtualDevice', 'managedDevices'],
    'Unit_Test': [
        'gradlew test', './gradlew test', 'testDebugUnitTest', 'testReleaseUnitTest',
        'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'
    ]
}

# === ACCURATE CI PLATFORM DETECTION ===
def detect_ci_platform(file_path, yaml_text):
    text = yaml_text.lower()
    file_lower = file_path.lower()

    # File path or filename patterns
    if '.circleci' in file_lower or 'circleci' in os.path.basename(file_lower):
        return "CircleCI"
    elif '.travis' in file_lower or 'travis' in os.path.basename(file_lower):
        return "Travis CI"
    elif '.gitlab' in file_lower or 'gitlab' in text or 'gitlab-ci' in text:
        return "GitLab CI"
    elif '.bitrise' in file_lower or 'bitrise' in text:
        return "Bitrise"
    elif '.github' in file_lower or 'github' in file_lower:
        return "GitHub Actions"
    elif 'github actions' in text:
        return "GitHub Actions"

    # Fallback based on common content markers
    if 'circleci' in text:
        return "CircleCI"
    if 'travis' in text:
        return "Travis CI"
    if 'bitrise' in text:
        return "Bitrise"
    if 'gitlab' in text or 'gitlab-ci' in text:
        return "GitLab CI"
    if 'github' in text:
        return "GitHub Actions"

    return "Unknown"

# === DETECT TEST TYPES USING KEYWORD SEARCH ===
def detect_testing_types(yaml_text):
    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()
    found = set()
    for label, keywords in TEST_TYPES.items():
        for kw in keywords:
            if kw.lower() in uncommented_text:
                found.add(label)
    return found

# === PROCESS YAML FILES ===
results = []

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            parts = filename.split(".")
            project_name = (parts[1] if len(parts) > 2 else parts[0]).lower()

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    raw = f.read().replace('\t', ' ')
                    ci_platform = detect_ci_platform(file_path, raw)
                    test_types = detect_testing_types(raw)

                    is_unit_test = 'Unit_Test' in test_types
                    filtered_types = []

                    for t in test_types:
                        if t == 'Unit_Test':
                            continue
                        elif t == 'emulator_manual':
                            filtered_types.append(f"{ci_platform}_emulator_manual")
                        else:
                            filtered_types.append(t)

                    test_type_str = ', '.join(sorted(filtered_types))
                    is_instr = bool(filtered_types)

                    results.append({
                        'project': project_name,
                        'ci_platform': ci_platform,
                        'test_type': test_type_str,
                        'unit_test': is_unit_test,
                        'instrumentation_test': is_instr
                    })

            except Exception:
                results.append({
                    'project': project_name,
                    'ci_platform': 'Error',
                    'test_type': '',
                    'unit_test': False,
                    'instrumentation_test': False
                })

# === EXPORT TO CSV ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ 7.1-YML_List.csv generated at: {OUTPUT_CSV}")


✅ 7.1-YML_List.csv generated at: C:\GitHub\Android-Mobile-Apps\7.1-YML_List.csv
